According to the original 15-checkpoint roadmap, we have completed through Checkpoint 7. Checkpoint 5.5 was an additional audit we inserted later, so it does not change the numbering.

That leaves 8 numbered notebooks/checkpoints: 8–15.

The remaining roadmap is:

Checkpoint	Purpose	New quantum simulation?
8	Causal-cone / effective-support mechanism	No
9	Entanglement-growth mechanism	Yes
10	Theory / 2-design / concentration consistency checks	Mostly no
11	Larger-system validation	Yes, potentially expensive
12	Cross-framework/Qiskit re-check	Yes
13	Noise robustness	Yes
14	Integrated model/evidence analysis	No
15	Final publication summary / conclusions	No

The exact ordering can be refined later, but 8 numbered checkpoints remain.

Notebook 8 — Causal-Cone / Effective-Support Analysis

This is the natural next step because Notebook 7 showed that connectivity changes the fitted onset behaviour. Now we test whether that difference is related to how rapidly the observable's influence spreads through the circuit.

Importantly, we will not assume beforehand that the causal cone must reach a particular fraction of the system at \(\tau_{BP}\). We measure it and let the data determine the relationship.

This notebook is deterministic graph analysis; it does not rerun the expensive quantum simulations.


### Cell 1 — Setup


In [ ]:
# ============================================================
# NOTEBOOK 8 — CAUSAL-CONE / EFFECTIVE-SUPPORT ANALYSIS
# ============================================================
#
# Scientific question:
#
#   Does the architecture-dependent tau_BP onset correlate
#   with the depth at which the observable's backward causal
#   cone expands substantially through the circuit?
#
# Architectures:
#   - brickwall_1d
#   - grid_2d
#   - all_to_all
#
# IMPORTANT:
#   - No quantum simulation.
#   - No fitted causal-cone threshold is assumed.
#   - tau_BP comes from the actual architecture experiment.
#   - Causal-cone growth is calculated deterministically from
#     the actual connectivity schedule.
#
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TAU_FILE = "architecture_tau.csv"
if not os.path.exists(TAU_FILE):
    alt_path = os.path.join("notebook7_architecture_generalization", "architecture_tau.csv")
    if os.path.exists(alt_path):
        TAU_FILE = alt_path

OUTPUT_DIR = "notebook8_causal_cone"
os.makedirs(OUTPUT_DIR, exist_ok=True)

THRESHOLD = 1e-2

ARCHITECTURES = [
    "brickwall_1d",
    "grid_2d",
    "all_to_all",
]

TRAIN_N = [8, 10, 12]
HELDOUT_N = [14]

K_BY_N = {
    8:  [2, 3, 4, 5, 6, 8],
    10: [2, 3, 4, 5, 6, 8, 10],
    12: [2, 3, 4, 5, 6, 8, 10, 12],
    14: [2, 3, 4, 5, 6, 8, 10, 12, 14],
}

GRID_SHAPES = {
    8:  (2, 4),
    10: (2, 5),
    12: (3, 4),
    14: (2, 7),
}

print("=" * 80)
print("NOTEBOOK 8 — CAUSAL-CONE / EFFECTIVE-SUPPORT ANALYSIS")
print("=" * 80)
print(f"Input tau dataset: {TAU_FILE}")
print(f"Threshold: {THRESHOLD}")
print(f"Architectures: {ARCHITECTURES}")
print("=" * 80)

### Cell 2 — Load actual \(\tau_{BP}\) results


In [ ]:
# ============================================================
# CELL 2 — LOAD ARCHITECTURE TAU RESULTS
# ============================================================

if not os.path.exists(TAU_FILE):
    raise FileNotFoundError(
        f"CRITICAL: {TAU_FILE} not found."
    )

tau_df = pd.read_csv(TAU_FILE)

required = [
    "architecture",
    "n",
    "k",
    "nk",
    "tau_BP",
    "censored",
]

missing = [
    c for c in required
    if c not in tau_df.columns
]

if missing:
    raise ValueError(
        f"CRITICAL: Missing columns: {missing}"
    )

for c in [
    "n",
    "k",
    "nk",
    "tau_BP",
]:

    tau_df[c] = pd.to_numeric(
        tau_df[c],
        errors="coerce"
    )

if tau_df[required].isna().any().any():
    raise ValueError(
        "CRITICAL: Missing/non-numeric values detected."
    )

print(
    f"Loaded {len(tau_df)} tau_BP observations."
)

display(
    tau_df.head(20)
)

### Cell 3 — Reconstruct the same connectivity schedules


In [ ]:
# ============================================================
# CELL 3 — CONNECTIVITY DEFINITIONS
# ============================================================
#
# Reconstruct the actual architecture schedules used in
# Notebook 7 rather than importing hidden state from that
# notebook.
#
# The causal-cone calculation only needs the two-qubit
# interaction graph at every circuit layer.
# ============================================================

def brickwall_pairs(
    n,
    layer
):

    if layer % 2 == 0:

        return [
            (q, q + 1)
            for q in range(
                0,
                n - 1,
                2
            )
        ]

    return [
        (q, q + 1)
        for q in range(
            1,
            n - 1,
            2
        )
    ]


def grid_edges(
    n
):

    if n not in GRID_SHAPES:
        raise ValueError(
            f"No grid shape for n={n}"
        )

    rows, cols = GRID_SHAPES[n]

    edges = []

    # Horizontal
    for r in range(rows):

        for c in range(cols - 1):

            q1 = r * cols + c
            q2 = r * cols + c + 1

            edges.append(
                (q1, q2)
            )

    # Vertical
    for r in range(rows - 1):

        for c in range(cols):

            q1 = r * cols + c
            q2 = (r + 1) * cols + c

            edges.append(
                (q1, q2)
            )

    return edges


def greedy_matchings(
    edges
):

    remaining = list(edges)
    matchings = []

    while remaining:

        used = set()
        current = []
        leftover = []

        for q1, q2 in remaining:

            if (
                q1 not in used
                and q2 not in used
            ):

                current.append(
                    (q1, q2)
                )

                used.add(q1)
                used.add(q2)

            else:

                leftover.append(
                    (q1, q2)
                )

        matchings.append(
            current
        )

        remaining = leftover

    return matchings


GRID_MATCHINGS = {
    n: greedy_matchings(
        grid_edges(n)
    )
    for n in GRID_SHAPES
}


def round_robin_matchings(
    n
):

    nodes = list(
        range(n)
    )

    if n % 2 == 1:
        nodes.append(None)

    fixed = nodes[-1]
    rotating = nodes[:-1]

    rounds = len(nodes) - 1

    matchings = []

    for _ in range(rounds):

        current_nodes = (
            [fixed]
            + rotating
        )

        pairs = []

        for i in range(
            len(current_nodes) // 2
        ):

            a = current_nodes[i]
            b = current_nodes[
                -1 - i
            ]

            if (
                a is not None
                and b is not None
            ):

                pairs.append(
                    (a, b)
                )

        matchings.append(
            pairs
        )

        rotating = (
            [rotating[-1]]
            + rotating[:-1]
        )

    return matchings


def architecture_pairs(
    architecture,
    n,
    layer
):

    if architecture == "brickwall_1d":

        return brickwall_pairs(
            n,
            layer
        )

    if architecture == "grid_2d":

        matchings = GRID_MATCHINGS[n]

        return matchings[
            layer % len(matchings)
        ]

    if architecture == "all_to_all":

        matchings = round_robin_matchings(
            n
        )

        return matchings[
            layer % len(matchings)
        ]

    raise ValueError(
        f"Unknown architecture: {architecture}"
    )

### Cell 4 — Backward causal-cone calculation


In [ ]:
# ============================================================
# CELL 4 — BACKWARD CAUSAL CONE
# ============================================================
#
# Definition:
#
# Start with the qubits supporting the observable.
#
# Traverse the circuit BACKWARD.
#
# If a two-qubit gate touches ANY qubit currently in the
# support, both qubits become part of the backward support.
#
# This gives the effective topological causal cone.
#
# NOTE:
# This tracks support propagation implied by connectivity.
# It does NOT claim that every gate produces a nonzero
# physical influence for every parameter realization.
# ============================================================

def backward_causal_cone(
    architecture,
    n,
    k,
    max_depth
):

    if k > n:
        raise ValueError(
            "k cannot exceed n."
        )

    support = set(
        range(k)
    )

    support_sizes = [
        len(support)
    ]

    newly_reached = [
        len(support)
    ]

    # Work backward from the final circuit layer.
    for depth in range(
        1,
        max_depth + 1
    ):

        forward_layer = (
            max_depth - depth
        )

        pairs = architecture_pairs(
            architecture,
            n,
            forward_layer
        )

        old_support = set(
            support
        )

        for q1, q2 in pairs:

            if (
                q1 in support
                or q2 in support
            ):

                support.add(q1)
                support.add(q2)

        support_sizes.append(
            len(support)
        )

        newly_reached.append(
            len(support)
            - len(old_support)
        )

    return (
        support_sizes,
        newly_reached
    )

### Cell 5 — Sanity-check cone monotonicity


In [ ]:
# ============================================================
# CELL 5 — CAUSAL-CONE SANITY CHECK
# ============================================================

print("=" * 80)
print("CAUSAL-CONE SANITY CHECK")
print("=" * 80)

cone_failures = []

for architecture in ARCHITECTURES:

    for n in GRID_SHAPES:

        for k in [
            2,
            min(4, n),
            n,
        ]:

            sizes, _ = backward_causal_cone(
                architecture,
                n,
                k,
                20
            )

            if np.any(
                np.diff(sizes) < 0
            ):

                cone_failures.append({
                    "architecture":
                        architecture,
                    "n":
                        n,
                    "k":
                        k,
                })

            if sizes[-1] > n:

                cone_failures.append({
                    "architecture":
                        architecture,
                    "n":
                        n,
                    "k":
                        k,
                    "reason":
                        "support > n",
                })

if cone_failures:

    display(
        pd.DataFrame(
            cone_failures
        )
    )

    raise RuntimeError(
        "CRITICAL: Causal-cone sanity check failed."
    )

print(
    "PASS: Causal-cone support is monotonic and bounded by n."
)

### Cell 6 — Calculate cone growth for every observed configuration


In [ ]:
# ============================================================
# CELL 6 — CONE GROWTH FOR ALL CONFIGURATIONS
# ============================================================

max_tau = int(
    tau_df["tau_BP"].max()
)

cone_rows = []

for _, row in tau_df.iterrows():

    architecture = row[
        "architecture"
    ]

    n = int(row["n"])
    k = int(row["k"])
    tau = int(row["tau_BP"])

    max_depth = max(
        max_tau,
        tau
    )

    sizes, newly_reached = (
        backward_causal_cone(
            architecture,
            n,
            k,
            max_depth
        )
    )

    for depth in range(
        max_depth + 1
    ):

        cone_rows.append({
            "architecture":
                architecture,

            "n":
                n,

            "k":
                k,

            "nk":
                n * k,

            "depth":
                depth,

            "cone_size":
                sizes[depth],

            "cone_fraction":
                sizes[depth] / n,

            "newly_reached":
                newly_reached[depth],

            "tau_BP":
                tau,
        })

cone_df = pd.DataFrame(
    cone_rows
)

print(
    f"Generated {len(cone_df):,} causal-cone observations."
)

display(
    cone_df.head(20)
)

### Cell 7 — Extract cone properties at \(\tau_{BP}\)


In [ ]:
# ============================================================
# CELL 7 — CONE STATE AT TAU_BP
# ============================================================

tau_cone_rows = []

for _, row in tau_df.iterrows():

    architecture = row[
        "architecture"
    ]

    n = int(row["n"])
    k = int(row["k"])
    tau = int(row["tau_BP"])

    subset = cone_df[
        (
            cone_df["architecture"]
            == architecture
        )
        &
        (
            cone_df["n"] == n
        )
        &
        (
            cone_df["k"] == k
        )
        &
        (
            cone_df["depth"] == tau
        )
    ]

    if len(subset) != 1:

        raise RuntimeError(
            "Could not locate unique cone state at tau_BP."
        )

    cone_state = subset.iloc[0]

    tau_cone_rows.append({
        "architecture":
            architecture,

        "n":
            n,

        "k":
            k,

        "nk":
            n * k,

        "tau_BP":
            tau,

        "cone_size_at_tau":
            int(cone_state["cone_size"]),

        "cone_fraction_at_tau":
            float(
                cone_state[
                    "cone_fraction"
                ]
            ),
    })

tau_cone_df = pd.DataFrame(
    tau_cone_rows
)

print("=" * 80)
print("CAUSAL-CONE STATE AT TAU_BP")
print("=" * 80)

display(
    tau_cone_df
)

### Cell 8 — Determine causal-cone saturation depth


In [ ]:
# ============================================================
# CELL 8 — CONE SATURATION DEPTH
# ============================================================

saturation_rows = []

for (
    architecture,
    n,
    k
), group in cone_df.groupby(
    [
        "architecture",
        "n",
        "k"
    ]
):

    group = group.sort_values(
        "depth"
    )

    full_system = group[
        group["cone_size"] >= n
    ]

    if len(full_system) > 0:

        saturation_depth = int(
            full_system.iloc[0]["depth"]
        )

    else:

        saturation_depth = np.nan

    saturation_rows.append({
        "architecture":
            architecture,

        "n":
            int(n),

        "k":
            int(k),

        "cone_saturation_depth":
            saturation_depth,
    })

saturation_df = pd.DataFrame(
    saturation_rows
)

tau_cone_df = tau_cone_df.merge(
    saturation_df,
    on=[
        "architecture",
        "n",
        "k"
    ],
    how="left"
)

tau_cone_df[
    "tau_minus_saturation_depth"
] = (
    tau_cone_df["tau_BP"]
    - tau_cone_df[
        "cone_saturation_depth"
    ]
)

display(
    tau_cone_df
)

### Cell 9 — Compare \(\tau_{BP}\) with causal-cone fraction


In [ ]:
# ============================================================
# CELL 9 — TAU VS CONE FRACTION
# ============================================================

print("=" * 80)
print("TAU_BP VS CAUSAL-CONE FRACTION")
print("=" * 80)

display(
    tau_cone_df[
        [
            "architecture",
            "n",
            "k",
            "tau_BP",
            "cone_size_at_tau",
            "cone_fraction_at_tau",
            "cone_saturation_depth",
            "tau_minus_saturation_depth",
        ]
    ]
)

### Cell 10 — Correlation analysis


In [ ]:
# ============================================================
# CELL 10 — CORRELATION ANALYSIS
# ============================================================

from scipy.stats import spearmanr

correlation_rows = []

for architecture in ARCHITECTURES:

    sub = tau_cone_df[
        tau_cone_df[
            "architecture"
        ] == architecture
    ].copy()

    # tau vs saturation depth
    valid_sat = sub.dropna(
        subset=[
            "tau_BP",
            "cone_saturation_depth"
        ]
    )

    if len(valid_sat) >= 3:

        rho_sat, p_sat = spearmanr(
            valid_sat["tau_BP"],
            valid_sat["cone_saturation_depth"]
        )

    else:

        rho_sat = np.nan
        p_sat = np.nan

    # tau vs cone fraction at onset
    if len(sub) >= 3:

        rho_fraction, p_fraction = spearmanr(
            sub["tau_BP"],
            sub["cone_fraction_at_tau"]
        )

    else:

        rho_fraction = np.nan
        p_fraction = np.nan

    correlation_rows.append({
        "architecture":
            architecture,

        "N":
            len(sub),

        "Spearman_tau_vs_cone_saturation":
            rho_sat,

        "p_tau_vs_cone_saturation":
            p_sat,

        "Spearman_tau_vs_cone_fraction":
            rho_fraction,

        "p_tau_vs_cone_fraction":
            p_fraction,
    })

correlation_df = pd.DataFrame(
    correlation_rows
)

print("=" * 80)
print("CAUSAL-CONE CORRELATIONS")
print("=" * 80)

display(
    correlation_df
)

### Cell 11 — Test whether onset occurs at a consistent cone fraction


In [ ]:
# ============================================================
# CELL 11 — ONSET CONE-FRACTION DISTRIBUTION
# ============================================================

fraction_summary = (
    tau_cone_df
    .groupby("architecture")
    .agg(
        N=("cone_fraction_at_tau", "size"),
        mean=("cone_fraction_at_tau", "mean"),
        median=("cone_fraction_at_tau", "median"),
        std=("cone_fraction_at_tau", "std"),
        minimum=("cone_fraction_at_tau", "min"),
        maximum=("cone_fraction_at_tau", "max"),
    )
    .reset_index()
)

print("=" * 80)
print("CONE FRACTION AT TAU_BP")
print("=" * 80)

display(
    fraction_summary
)

### Cell 12 — Plot cone growth


In [ ]:
# ============================================================
# CELL 12 — REPRESENTATIVE CONE GROWTH
# ============================================================

representatives = [
    (8, 2),
    (10, 4),
    (14, 2),
]

for n, k in representatives:

    plt.figure(
        figsize=(9, 6)
    )

    for architecture in ARCHITECTURES:

        sub = cone_df[
            (
                cone_df["architecture"]
                == architecture
            )
            &
            (
                cone_df["n"] == n
            )
            &
            (
                cone_df["k"] == k
            )
        ]

        if len(sub) == 0:
            continue

        plt.plot(
            sub["depth"],
            sub["cone_fraction"],
            marker="o",
            markersize=3,
            label=architecture
        )

        tau_row = tau_df[
            (
                tau_df["architecture"]
                == architecture
            )
            &
            (
                tau_df["n"] == n
            )
            &
            (
                tau_df["k"] == k
            )
        ]

        if len(tau_row) == 1:

            tau = int(
                tau_row.iloc[0]["tau_BP"]
            )

            plt.axvline(
                tau,
                linestyle="--",
                alpha=0.5
            )

    plt.axhline(
        1.0,
        linestyle=":",
        label="Full-system support"
    )

    plt.xlabel(
        "Circuit depth"
    )

    plt.ylabel(
        "Backward causal-cone fraction"
    )

    plt.title(
        f"Causal-Cone Growth: n={n}, k={k}"
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"cone_growth_n{n}_k{k}.png"
        ),
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()

### Cell 13 — Plot \(\tau_{BP}\) vs cone saturation depth


In [ ]:
# ============================================================
# CELL 13 — TAU VS CONE SATURATION
# ============================================================

plt.figure(
    figsize=(9, 7)
)

for architecture in ARCHITECTURES:

    sub = tau_cone_df[
        (
            tau_cone_df[
                "architecture"
            ]
            == architecture
        )
        &
        (
            tau_cone_df[
                "cone_saturation_depth"
            ].notna()
        )
    ]

    plt.scatter(
        sub["cone_saturation_depth"],
        sub["tau_BP"],
        s=60,
        label=architecture
    )

all_valid = tau_cone_df[
    tau_cone_df[
        "cone_saturation_depth"
    ].notna()
]

if len(all_valid) > 0:

    lo = min(
        all_valid[
            "cone_saturation_depth"
        ].min(),
        all_valid[
            "tau_BP"
        ].min()
    )

    hi = max(
        all_valid[
            "cone_saturation_depth"
        ].max(),
        all_valid[
            "tau_BP"
        ].max()
    )

    plt.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=1.5,
        label="tau_BP = cone saturation depth"
    )

plt.xlabel(
    "Depth of full causal-cone saturation"
)

plt.ylabel(
    r"$\tau_{BP}$"
)

plt.title(
    "Barren-Plateau Onset vs Causal-Cone Saturation"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "tau_vs_cone_saturation.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 14 — Plot onset cone fraction


In [ ]:
# ============================================================
# CELL 14 — CONE FRACTION AT ONSET
# ============================================================

plt.figure(
    figsize=(9, 6)
)

positions = {
    architecture: i
    for i, architecture
    in enumerate(
        ARCHITECTURES
    )
}

for architecture in ARCHITECTURES:

    sub = tau_cone_df[
        tau_cone_df[
            "architecture"
        ] == architecture
    ]

    x = np.full(
        len(sub),
        positions[architecture],
        dtype=float
    )

    jitter = np.linspace(
        -0.12,
        0.12,
        len(sub)
    )

    plt.scatter(
        x + jitter,
        sub["cone_fraction_at_tau"],
        s=55
    )

plt.xticks(
    list(positions.values()),
    list(positions.keys())
)

plt.ylabel(
    "Causal-cone fraction at tau_BP"
)

plt.xlabel(
    "Architecture"
)

plt.title(
    "Effective Causal Support at Barren-Plateau Onset"
)

plt.grid(
    True,
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "cone_fraction_at_tau.png"
    ),
    dpi=250,
    bbox_inches="tight"
)

plt.show()

### Cell 15 — Explicit mechanism audit


In [ ]:
# ============================================================
# CELL 15 — MECHANISM AUDIT
# ============================================================
#
# We intentionally do NOT impose a preselected cone fraction.
#
# This cell produces descriptive statistics only.
# ============================================================

print("\n" + "=" * 90)
print("CHECKPOINT 8 — CAUSAL-CONE MECHANISM AUDIT")
print("=" * 90)

print(
    "\nArchitecture-level summary:"
)

display(
    fraction_summary
)

print(
    "\nCorrelation of tau_BP with causal-cone saturation:"
)

display(
    correlation_df[
        [
            "architecture",
            "N",
            "Spearman_tau_vs_cone_saturation",
            "p_tau_vs_cone_saturation",
            "Spearman_tau_vs_cone_fraction",
            "p_tau_vs_cone_fraction",
        ]
    ]
)

print(
    "\nInterpretation rule:"
)

print(
    "A strong association would support causal-cone expansion "
    "as a candidate structural correlate of the onset."
)

print(
    "It does NOT by itself establish causality."
)

print(
    "Differences between architectures must be interpreted "
    "together with the actual connectivity and the finite-depth "
    "definition of tau_BP."
)

### Cell 16 — Save results


In [ ]:
# ============================================================
# CELL 16 — SAVE CHECKPOINT 8
# ============================================================

cone_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "causal_cone_growth.csv"
    ),
    index=False
)

tau_cone_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "tau_causal_cone_metrics.csv"
    ),
    index=False
)

saturation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cone_saturation_depth.csv"
    ),
    index=False
)

fraction_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cone_fraction_summary.csv"
    ),
    index=False
)

correlation_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cone_correlation_analysis.csv"
    ),
    index=False
)

protocol = {
    "checkpoint": 8,
    "experiment":
        "causal_cone_effective_support",

    "architectures":
        ARCHITECTURES,

    "threshold":
        THRESHOLD,

    "grid_shapes":
        GRID_SHAPES,

    "observable_support":
        "first k qubits",

    "causal_cone_definition":
        "Backward support propagation through every "
        "two-qubit interaction layer",

    "tau_source":
        TAU_FILE,

    "no_quantum_simulation":
        True,

    "no_preselected_cone_threshold":
        True,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "checkpoint8_protocol.json"
    ),
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

print("=" * 80)
print("CHECKPOINT 8 OUTPUTS SAVED")
print("=" * 80)

for fname in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        os.path.join(
            OUTPUT_DIR,
            fname
        )
    )

print(
    "\nSTATUS: COMPLETE"
)

What we are looking for

The key outputs are Cell 11 and Cell 15.

We are specifically testing whether the architecture differences seen in Notebook 7 correspond to different rates of backward-support growth, rather than simply declaring that they do.

One important distinction: this notebook measures a topological causal cone, not the actual numerical magnitude of influence. So if the correlation is imperfect, that does not invalidate the idea; it tells us that connectivity alone does not fully explain the gradient-variance onset.

Run Notebook 8 and send me the Cell 15 output.
